# Avaliacao de Naive Bayes
## Imports e funções

In [1]:
import pandas as pd
import numpy as np
from numpy import mean
from numpy import std
from sklearn import metrics
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict
from sklearn.naive_bayes import GaussianNB, MultinomialNB, ComplementNB

def raca_para_especie(raca):
    if raca in ['basset_hound', 'saint_bernard']:
        return 'dog'
    elif raca in ['Birman', 'Persian']:
        return 'cat'
    else:
        return raca  # fallback

### Para Bases com PCA que geram valores negativos
from sklearn.preprocessing import minmax_scale

def separar_dataset(df, scale=False):
    X = df.iloc[:, :-1]
    if scale:
        X = minmax_scale(X)
    y = df.iloc[:, -1]
    return X, y




## Lendo lista de arquivos a processar

In [2]:
datafiles = pd.read_csv('dataset_list.csv',encoding='utf-8')

datafiles.head(12)

,key,filename
0,hogfeat_128_16_4_9_pca,../Aula11/hogfeat_128_16_4_9_pca.csv.gz
1,hogfeat_256_64_2_18,../Aula11/hogfeat_256_64_2_18.csv.gz
2,hogfeat_256_64_2_9,../Aula11/hogfeat_256_64_2_9.csv.gz
3,hogfeat_128_16_4_9,../Aula11/hogfeat_128_16_4_9.csv.gz
4,hogfeat_256_32_2_9_pca,../Aula11/hogfeat_256_32_2_9_pca.csv.gz
5,hogfeat_128_16_2_9_pca,../Aula11/hogfeat_128_16_2_9_pca.csv.gz
6,hogfeat_128_32_2_9,../Aula11/hogfeat_128_32_2_9.csv.gz
7,lbpfeat_256_6_48,../Aula11/lbpfeat_256_6_48.csv.gz
8,lbpfeat_256_12_96,../Aula11/lbpfeat_256_12_96.csv.gz
9,hogfeat_256_32_2_9,../Aula11/hogfeat_256_32_2_9.csv.gz


## Lendo Dataframes e ajustando dados

In [3]:

dfs = {}
shapes = []
last_cols = []

for metadata in datafiles.itertuples():
    # Imprime o arquivo que está sendo lido
    #print(f"Lendo arquivo: {metadata.key}")
    # Carrega o DataFrame
    df = pd.read_csv(metadata.filename)
    
    # Aplica a função raca_para_especie na coluna raca
    if 'raca' in df.columns:
        df['especie'] = df['raca'].apply(raca_para_especie)
        df = df.drop('raca', axis=1)
    
    # Elimina a coluna nome_arquivo se existir
    if 'nome_arquivo' in df.columns:
        df = df.drop('nome_arquivo', axis=1)
    
    dfs[metadata.key] = df
    shapes.append(df.shape)
    # Pega as últimas duas colunas
    last_cols.append(df.columns[-1:].tolist())

# Adiciona as colunas shape e last_two_columns ao datafiles
datafiles['shape'] = shapes
datafiles['last_column'] = last_cols

datafiles
    

,key,filename,shape,last_column
0,hogfeat_128_16_4_9_pca,../Aula11/hogfeat_128_16_4_9_pca.csv.gz,"(800, 104)",[especie]
1,hogfeat_256_64_2_18,../Aula11/hogfeat_256_64_2_18.csv.gz,"(800, 649)",[especie]
2,hogfeat_256_64_2_9,../Aula11/hogfeat_256_64_2_9.csv.gz,"(800, 325)",[especie]
3,hogfeat_128_16_4_9,../Aula11/hogfeat_128_16_4_9.csv.gz,"(800, 3601)",[especie]
4,hogfeat_256_32_2_9_pca,../Aula11/hogfeat_256_32_2_9_pca.csv.gz,"(800, 94)",[especie]
5,hogfeat_128_16_2_9_pca,../Aula11/hogfeat_128_16_2_9_pca.csv.gz,"(800, 118)",[especie]
6,hogfeat_128_32_2_9,../Aula11/hogfeat_128_32_2_9.csv.gz,"(800, 325)",[especie]
7,lbpfeat_256_6_48,../Aula11/lbpfeat_256_6_48.csv.gz,"(800, 51)",[especie]
8,lbpfeat_256_12_96,../Aula11/lbpfeat_256_12_96.csv.gz,"(800, 99)",[especie]
9,hogfeat_256_32_2_9,../Aula11/hogfeat_256_32_2_9.csv.gz,"(800, 1765)",[especie]


## Percorrendo os datasets para os tres criterios e tipos de treinamento pré-definidos

In [4]:
# Montando Gridsearch
par_criterios = [GaussianNB, MultinomialNB, ComplementNB]
par_training = ["holdout", "crossvalidation"]

# Estrutura para armazenar resultados
resultados = []

for metadata in datafiles.itertuples():
    print(f"Dataset: {metadata.key}")
    
    # Obtém o dataset correspondente do dicionário dfs
    df = dfs[metadata.key]
    X, y = separar_dataset(df, scale=metadata.key.endswith('pca'))
    
    for criterio in par_criterios:
        print(f"\tCriterio: {criterio.__name__}")
        for training in par_training:
            print(f"\t\tTraining: {training}")
            naive_bayes = criterio()
            f1 = 0.0
            f1_std = 0.0
            confusao = np.array([])
            
            if training == "holdout":
                # Treinamento e avaliação usando holdout
                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
                naive_bayes.fit(X_train, y_train)
                y_pred = naive_bayes.predict(X_test)
                f1 = f1_score(y_test, y_pred, average='weighted')
                f1_std = 0.0
                confusao = confusion_matrix(y_test, y_pred)
            elif training == "crossvalidation":
                # Treinamento e avaliação usando crossvalidation
                kf = KFold(n_splits=10, random_state=42, shuffle=True)
                scores = cross_val_score(naive_bayes, X, y, scoring='f1_weighted', cv=kf)
                y_pred = cross_val_predict(naive_bayes, X, y, cv=kf)
                confusao = confusion_matrix(y, y_pred)
                f1 = scores.mean()
                f1_std = scores.std()
            else:
                print(f"\t\t\tTraining type {training} nao suportado")
                continue
            
            # Armazena resultados no DataFrame
            resultado = {
                'dataset': metadata.key,
                'criterio': criterio.__name__,
                'training_type': training,
                'f1_score': f1,
                'f1_std': f1_std,
                'confusion_matrix': confusao,
                'model': naive_bayes
            }
            resultados.append(resultado)
            print('\t\t\tF1 Score: %.3f (%.3f)' % (f1, f1_std))
            # print(f"\t\t\tf1_score: {f1}, f1_std: {f1_std}")
    print("-" * 50)

# Converte para DataFrame
df_resultados = pd.DataFrame(resultados)
                
df_resultados['shape'] = df_resultados['dataset'].apply(lambda x: datafiles.loc[datafiles['key'] == x, 'shape'].values[0])



Dataset: hogfeat_128_16_4_9_pca
	Criterio: GaussianNB
		Training: holdout
			F1 Score: 0.713 (0.000)
		Training: crossvalidation
			F1 Score: 0.706 (0.035)
	Criterio: MultinomialNB
		Training: holdout
			F1 Score: 0.471 (0.000)
		Training: crossvalidation
			F1 Score: 0.723 (0.058)
	Criterio: ComplementNB
		Training: holdout
			F1 Score: 0.776 (0.000)
		Training: crossvalidation
			F1 Score: 0.774 (0.020)
--------------------------------------------------
Dataset: hogfeat_256_64_2_18
	Criterio: GaussianNB
		Training: holdout
			F1 Score: 0.755 (0.000)
		Training: crossvalidation
			F1 Score: 0.736 (0.029)
	Criterio: MultinomialNB
		Training: holdout
			F1 Score: 0.738 (0.000)
		Training: crossvalidation
			F1 Score: 0.724 (0.042)
	Criterio: ComplementNB
		Training: holdout
			F1 Score: 0.734 (0.000)
		Training: crossvalidation
			F1 Score: 0.723 (0.036)
--------------------------------------------------
Dataset: hogfeat_256_64_2_9
	Criterio: GaussianNB
		Training: holdout
			F1 Score: 

## Salvando resultados em CSV e Joblib

In [5]:
# Adiciona coluna shape no df_resultados buscando no datafiles
df_resultados['shape'] = df_resultados['dataset'].apply(lambda x: datafiles.loc[datafiles['key'] == x, 'shape'].values[0])

import joblib
joblib.dump(df_resultados, 'naive_bayes_avaliacao_resultados.joblib')

df_csv = df_resultados.drop(['confusion_matrix', 'model'], axis=1)
df_csv.to_csv('naive_bayes_avaliacao_resultados.csv', index=False)

### Amostra do DataFrame de Resultados

In [7]:
# Visualiza os resultados
print(f"Total de combinações processadas: {len(df_resultados)}")
print(f"Colunas disponíveis: {df_resultados.columns.tolist()}")
df_resultados


Total de combinações processadas: 72
Colunas disponíveis: ['dataset', 'criterio', 'training_type', 'f1_score', 'f1_std', 'confusion_matrix', 'model', 'shape']


,dataset,criterio,training_type,f1_score,f1_std,confusion_matrix,model,shape
0,hogfeat_128_16_4_9_pca,GaussianNB,holdout,0.712635,0.000000,"[[85, 21], [48, 86]]",GaussianNB(),"(800, 104)"
1,hogfeat_128_16_4_9_pca,GaussianNB,crossvalidation,0.705807,0.034680,"[[293, 107], [127, 273]]",GaussianNB(),"(800, 104)"
2,hogfeat_128_16_4_9_pca,MultinomialNB,holdout,0.471084,0.000000,"[[105, 1], [108, 26]]",MultinomialNB(),"(800, 104)"
3,hogfeat_128_16_4_9_pca,MultinomialNB,crossvalidation,0.722614,0.057761,"[[286, 114], [105, 295]]",MultinomialNB(),"(800, 104)"
4,hogfeat_128_16_4_9_pca,ComplementNB,holdout,0.775752,0.000000,"[[87, 19], [35, 99]]",ComplementNB(),"(800, 104)"
...,...,...,...,...,...,...,...,...
67,lbpfeat_256_24_192,GaussianNB,crossvalidation,0.692671,0.039264,"[[235, 165], [78, 322]]",GaussianNB(),"(800, 195)"
68,lbpfeat_256_24_192,MultinomialNB,holdout,0.270617,0.000000,"[[106, 0], [134, 0]]",MultinomialNB(),"(800, 195)"
69,lbpfeat_256_24_192,MultinomialNB,crossvalidation,0.348546,0.123475,"[[49, 351], [58, 342]]",MultinomialNB(),"(800, 195)"
70,lbpfeat_256_24_192,ComplementNB,holdout,0.471470,0.000000,"[[9, 97], [3, 131]]",ComplementNB(),"(800, 195)"
